In [4]:
from align import extract_word_embeddings
import torch

all_tokens = [
    ["I", "love", "Paris", "butterfly"],
    ["New", "York", "is", "huge"],
    ["Hello", "world"]
]

all_labels = [
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [0, 0]
]

# Extract embeddings
embeddings_list, labels = extract_word_embeddings(
    model=None,
    tokenizer=None,
    all_tokens=all_tokens,
    all_labels=all_labels,
    model_name="bert-base-cased",
    device="cuda"  # Use GPU if available
)

# Verify alignment
print("\n" + "=" * 50)
print("Alignment check:")
for i, (embs, lbls) in enumerate(zip(embeddings_list, labels)):
    aligned = embs.shape[0] == len(lbls)
    print(f"Sentence {i+1}: embeddings={embs.shape[0]}, labels={len(lbls)} -> {'OK' if aligned else 'MISMATCH'}")

# Save results
torch.save({
    "embeddings": embeddings_list,
    "labels": labels
}, "word_embeddings.pt")
print("\n💾 Saved to 'word_embeddings.pt'")




Using device: cuda
Processing sentence 1/3: I love Paris butterfly
  -> Got 4 word embeddings
Processing sentence 2/3: New York is huge
  -> Got 4 word embeddings
Processing sentence 3/3: Hello world
  -> Got 2 word embeddings

✅ Finished. Total sentences processed: 3

Alignment check:
Sentence 1: embeddings=4, labels=4 -> OK
Sentence 2: embeddings=4, labels=4 -> OK
Sentence 3: embeddings=2, labels=2 -> OK

💾 Saved to 'word_embeddings.pt'


In [24]:
from align2 import extract_word_embeddings
all_tokens = [
    ["I", "love", "Paris", "butter+fly","."],
    ["New", "York", "is", "big"],
    ["Transformer", "models", "are", "great"]
]

all_labels = [
    [0, 0, 1, 1,0],
    [0, 0, 0, 1],
    [1, 0, 0, 0]
]

# Extract with token alignment
aligned_data = extract_word_embeddings(
        model=None,
        tokenizer=None,
        all_tokens=all_tokens,
        all_labels=all_labels,
        model_name="bert-base-cased",
        device="cuda"
    )

# Example: Access results
print("\n" + "="*60)
print("Final aligned data (first sentence):")
for item in aligned_data[0]:
    print(f"Token: '{item['token']}', Label: {item['label']}, Embedding shape: {item['embedding'].shape}")

Using device: cuda

Sentence 1/3: I love Paris butter+fly .
Tokenization alignment:
----------------------------------------
  [Special] [CLS]      -> position 0 (ignored)
  Token  0 'I         ' -> subword 'I         ' at position 1 (used)
  Token  1 'love      ' -> subword 'love      ' at position 2 (used)
  Token  2 'Paris     ' -> subword 'Paris     ' at position 3 (used)
  Token  3 'butter+fly' -> subword 'butter    ' at position 4 (used)
                          subword '+         ' at position 5 (skipped)
                          subword 'fly       ' at position 6 (skipped)
  Token  4 '.         ' -> subword '.         ' at position 7 (used)
  [Special] [SEP]      -> position 8 (ignored)
✅ Extracted 5 word embeddings for this sentence.

Sentence 2/3: New York is big
Tokenization alignment:
----------------------------------------
  [Special] [CLS]      -> position 0 (ignored)
  Token  0 'New       ' -> subword 'New       ' at position 1 (used)
  Token  1 'York      ' -> subwor

In [14]:
# 第1个句子，第2个词的 token 和 embedding
token = aligned_data[0][1]['token']        # 例如 'love'
emb   = aligned_data[0][1]['embedding']    # tensor(768,)
label = aligned_data[0][1]['label']        # 例如 0
label

0

In [20]:
first_sentence_embeddings = [item['embedding'] for item in aligned_data[0]]
len(first_sentence_embeddings[0])

768

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# 模拟数据
X_train = np.random.rand(1000, 768)  # 1000 个正常文本块
X_test  = np.random.rand(50, 768)    # 5 个测试文本，每文本 10 块

# 一行 fit，一行 predict
nbrs = NearestNeighbors(n_neighbors=1).fit(X_train)
patch_distances, _ = nbrs.kneighbors(X_test)
patch_distances = patch_distances.flatten()  # (50,)

# 转成图像级分数（每10个patch是一张图）
patch_distances = patch_distances.reshape(-1, 10)  # (5, 10)
image_scores = np.max(patch_distances, axis=1)     # (5,)

print("图像异常分数:", image_scores)